# Notebook 4: Directional Consistency (GSE39099)
## CausalPMOS-Map — Directional Support D(g)

**Goal:** Assess whether gene expression changes follow a consistent direction
across the normal → hyperplasia → cancer progression. This feeds the
**Directional component D(g)** of the PCBS score.

**Dataset:** [GSE39099](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE39099)  
- **4 pooled arrays** on GPL570 (Affymetrix HG-U133 Plus 2.0)  
- Each array = pool of 10 patients (40 patients total, pooled into 4 arrays)  
- 4 groups (1 array each): Normal endometrium, Atypical hyperplasia,
  Early-stage cancer, Advanced-stage cancer  

**⚠ IMPORTANT — Pooled arrays:** Each array represents a
**pool of 10 patients**, not an independent biological replicate.
With only 1 array per group there is **no biological replication**,
so standard statistical tests (limma, t-test, etc.) are invalid.
This notebook computes **fold-changes** and **directional trends**
across stages only — no p-values.

**Methods:**
- Expression per stage from single pooled array (log₂ scale)  
- Fold-change between adjacent stages: Normal→Hyperplasia, Hyperplasia→Cancer(early), Cancer(early)→Cancer(advanced)  
- Monotonic trend detection: does the gene move consistently up or down across the full progression?  
- Cross-reference with NB1/NB2/NB3 DEG directions for concordance  

**Key output:** `results/directional_support_gse39099.csv`

## 1. Setup

In [ ]:
# ── 1 · Install packages & imports ─────────────────────────────
!pip install -q GEOparse adjustText

import warnings, os
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print('Setup complete.')

## 2. Download GSE39099 via GEOparse

GEOparse downloads the SOFT file and provides both expression data
and GPL570 platform annotation for probe → gene symbol mapping.

In [ ]:
# ── 2 · Download dataset ───────────────────────────────────────
gse = GEOparse.get_GEO('GSE39099', destdir='./geo_cache', silent=True)

print(f'Title : {gse.metadata["title"][0]}')
print(f'Samples: {len(gse.gsms)}')
print(f'Platform: {list(gse.gpls.keys())}')

## 3. Sample Classification

GSE39099 has only **4 GSM arrays** (each a pool of 10 patients).
The sample titles are generic ("Endometrial cancer in different stages-1/2/3/4"),
so we hardcode the mapping from the GEO series description.

In [ ]:
# ── 3 · Hardcoded sample mapping (pooled arrays) ──────────────
# GSE39099 series description assigns stages 1-4 as:
#   stages-1 = Normal endometrium
#   stages-2 = Atypical hyperplasia
#   stages-3 = Early-stage endometrial cancer
#   stages-4 = Advanced-stage endometrial cancer

manual_groups = {
    'GSM956132': 'Normal',
    'GSM956133': 'Hyperplasia',
    'GSM956134': 'Cancer_Early',
    'GSM956135': 'Cancer_Advanced',
}

sample_info = []
for gsm_name, gsm in gse.gsms.items():
    sample_info.append({
        'GSM': gsm_name,
        'title': gsm.metadata.get('title', [''])[0],
        'group': manual_groups.get(gsm_name, 'Unknown'),
    })

sample_df = pd.DataFrame(sample_info)

print(f'Total arrays: {len(sample_df)}  (each = pool of 10 patients)')
print()
print(sample_df[['GSM', 'title', 'group']].to_string(index=False))

In [ ]:
# ── 3b · Verify all 4 groups present ──────────────────────────
expected = {'Normal', 'Hyperplasia', 'Cancer_Early', 'Cancer_Advanced'}
found = set(sample_df['group'].unique())
missing = expected - found

if missing:
    raise ValueError(f'Missing groups: {missing} — check manual_groups mapping')
else:
    print('All 4 groups found ✓')
    for g in ['Normal', 'Hyperplasia', 'Cancer_Early', 'Cancer_Advanced']:
        n = (sample_df['group'] == g).sum()
        print(f'  {g}: {n} array (pool of 10 patients)')

## 4. Probe-to-Gene Mapping (GPL570)

In [ ]:
# ── 4 · Build expression matrix + map probes to genes ──────────

# Get GPL570 annotation
gpl_name = list(gse.gpls.keys())[0]
gpl = gse.gpls[gpl_name]
annot = gpl.table

# Find gene symbol column
sym_col = None
for c in annot.columns:
    if 'gene' in c.lower() and 'symbol' in c.lower():
        sym_col = c; break
if sym_col is None:
    for c in annot.columns:
        if c.upper() == 'GENE_SYMBOL' or c.upper() == 'SYMBOL':
            sym_col = c; break
if sym_col is None:
    # Fallback: try 'Gene Symbol' exactly
    sym_col = 'Gene Symbol' if 'Gene Symbol' in annot.columns else None

print(f'GPL: {gpl_name}')
print(f'Annotation rows: {len(annot):,}')
print(f'Gene symbol column: {sym_col}')

# Build probe → gene map (keep only probes with a valid gene symbol)
probe2gene = annot[['ID', sym_col]].dropna(subset=[sym_col]).copy()
probe2gene = probe2gene[probe2gene[sym_col].astype(str).str.strip() != '']
probe2gene = probe2gene[~probe2gene[sym_col].str.contains('///')]
probe2gene.columns = ['Probe', 'Gene']
probe2gene['Gene'] = probe2gene['Gene'].str.strip()
print(f'Usable probe→gene mappings: {len(probe2gene):,}')
print(f'Unique genes: {probe2gene["Gene"].nunique():,}')

In [ ]:
# ── 4b · Build expression matrix ──────────────────────────────

# Extract expression values from each GSM
expr_frames = []
for gsm_name, gsm in gse.gsms.items():
    tbl = gsm.table
    # Find the value column (usually 'VALUE')
    val_col = 'VALUE' if 'VALUE' in tbl.columns else tbl.columns[1]
    s = tbl.set_index('ID_REF')[val_col].astype(float)
    s.name = gsm_name
    expr_frames.append(s)

expr_raw = pd.concat(expr_frames, axis=1)
print(f'Raw expression: {expr_raw.shape[0]:,} probes × {expr_raw.shape[1]} samples')

# Map probes to genes — take max-mean probe per gene
expr_raw.index.name = 'Probe'
expr_mapped = expr_raw.reset_index().merge(probe2gene, left_on='Probe', right_on='Probe')
sample_cols = [c for c in expr_mapped.columns if c.startswith('GSM')]

# For genes with multiple probes, keep the probe with highest mean expression
expr_mapped['_mean'] = expr_mapped[sample_cols].mean(axis=1)
idx_best = expr_mapped.groupby('Gene')['_mean'].idxmax()
expr_gene = expr_mapped.loc[idx_best].set_index('Gene')[sample_cols].copy()
expr_gene = expr_gene.drop(columns=['_mean'], errors='ignore')

print(f'After probe→gene mapping: {expr_gene.shape[0]:,} genes × {expr_gene.shape[1]} samples')

## 5. Quality Check

With only 4 arrays (1 per group), PCA is not meaningful.
Instead, we show a **sample-to-sample correlation heatmap** to verify
that the arrays are high-quality and that adjacent stages are more similar.

In [ ]:
# ── 5 · Sample-to-sample correlation heatmap ──────────────────

# Log2-transform if not already (GEO microarray data is often already log2)
if expr_gene.max().max() > 50:  # likely raw intensity
    expr_log = np.log2(expr_gene.clip(lower=1))
    print('Applied log₂ transform (raw intensities detected)')
else:
    expr_log = expr_gene.copy()
    print('Data appears already log₂-transformed')

# Compute pairwise Pearson correlation
ordered_samples = sample_df['GSM'].tolist()
ordered_samples = [s for s in ordered_samples if s in expr_log.columns]
corr = expr_log[ordered_samples].corr()

# Rename for readability
gsm_to_group = dict(zip(sample_df['GSM'], sample_df['group']))
labels = [f'{gsm_to_group[s]}\n({s})' for s in ordered_samples]
corr.index = labels
corr.columns = labels

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0.85, vmax=1.0,
            ax=ax, square=True, cbar_kws={'label': 'Pearson r'})
ax.set_title('Sample-to-Sample Correlation (4 Pooled Arrays)')
plt.tight_layout()
plt.savefig('figures/nb4_01_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Each array is a pool of 10 patients — no biological replicates.')

## 6. Group-Mean Fold Changes

Since each array is a pool of 10 patients, we compute the **mean
expression per group** and derive **log₂ fold-changes** between
adjacent stages:

1. Normal → Hyperplasia  
2. Hyperplasia → Early Cancer  
3. Early Cancer → Advanced Cancer  

No p-values are computed — pooled data violates the independence
assumption of standard tests.

In [ ]:
# ── 6 · Compute group means and fold changes ──────────────────

group_ids = {}
for grp in ['Normal', 'Hyperplasia', 'Cancer_Early', 'Cancer_Advanced']:
    ids = sample_df.loc[sample_df['group'] == grp, 'GSM'].tolist()
    ids = [s for s in ids if s in expr_log.columns]
    group_ids[grp] = ids
    print(f'{grp}: {len(ids)} array (pooled, 10 patients)')

# Expression per group (with 1 array each, "mean" is just the single value)
means = pd.DataFrame({
    'mean_Normal':           expr_log[group_ids['Normal']].mean(axis=1),
    'mean_Hyperplasia':      expr_log[group_ids['Hyperplasia']].mean(axis=1),
    'mean_Cancer_Early':     expr_log[group_ids['Cancer_Early']].mean(axis=1),
    'mean_Cancer_Advanced':  expr_log[group_ids['Cancer_Advanced']].mean(axis=1),
})

# Log₂ fold-changes between adjacent stages
means['logFC_Normal_to_Hyper']    = means['mean_Hyperplasia']     - means['mean_Normal']
means['logFC_Hyper_to_EarlyCa']   = means['mean_Cancer_Early']    - means['mean_Hyperplasia']
means['logFC_EarlyCa_to_AdvCa']   = means['mean_Cancer_Advanced'] - means['mean_Cancer_Early']

# Overall: Normal → Advanced Cancer
means['logFC_Normal_to_AdvCa']    = means['mean_Cancer_Advanced'] - means['mean_Normal']

means.index.name = 'Gene'
print(f'\nComputed fold-changes for {len(means):,} genes')
print(means.head())

## 7. Monotonic Trend Detection

A gene has a **monotonically increasing** (or decreasing) trend if
its expression moves in the same direction at every stage transition.
This is the strongest directional evidence.

In [ ]:
# ── 7 · Classify directional trend ─────────────────────────────

def classify_trend(row):
    """Classify the 3-step fold-change pattern."""
    fc1 = row['logFC_Normal_to_Hyper']
    fc2 = row['logFC_Hyper_to_EarlyCa']
    fc3 = row['logFC_EarlyCa_to_AdvCa']
    
    signs = (np.sign(fc1), np.sign(fc2), np.sign(fc3))
    
    if signs == (1, 1, 1):
        return 'Monotonic_Up'
    elif signs == (-1, -1, -1):
        return 'Monotonic_Down'
    elif signs[0] == signs[1] and signs[0] != 0:
        # First two steps agree — mostly consistent
        direction = 'Up' if signs[0] > 0 else 'Down'
        return f'Partial_{direction}'       # up/down then reversal
    elif signs[1] == signs[2] and signs[1] != 0:
        direction = 'Up' if signs[1] > 0 else 'Down'
        return f'Late_{direction}'          # late-onset trend
    else:
        return 'Non_monotonic'

means['Trend'] = means.apply(classify_trend, axis=1)

print('Directional trends across Normal → Hyperplasia → Early Ca → Advanced Ca:')
print(means['Trend'].value_counts().to_string())

In [ ]:
# ── 7b · Compute D(g) directional support score ───────────────

def compute_dg(row):
    """
    D(g) = directional consistency score for PCBS.
    
    Scores the agreement of the 3 stage transitions:
      - 3/3 same sign (monotonic) → 1.0
      - 2/3 same sign (partial)   → 0.67
      - 1/3 or mixed              → 0.33
      - all zero                   → 0.0
    
    Sign indicates overall direction:
      positive D(g) → gene trends upward across progression
      negative D(g) → gene trends downward across progression
    """
    fc1 = row['logFC_Normal_to_Hyper']
    fc2 = row['logFC_Hyper_to_EarlyCa']
    fc3 = row['logFC_EarlyCa_to_AdvCa']
    
    signs = [np.sign(fc1), np.sign(fc2), np.sign(fc3)]
    nonzero = [s for s in signs if s != 0]
    
    if len(nonzero) == 0:
        return 0.0
    
    n_pos = sum(1 for s in nonzero if s > 0)
    n_neg = sum(1 for s in nonzero if s < 0)
    majority_dir = 1 if n_pos >= n_neg else -1
    n_agree = max(n_pos, n_neg)
    
    # Consistency fraction (out of 3 transitions)
    consistency = n_agree / 3.0
    
    # Weight by magnitude of overall change (larger = more confident)
    overall_fc = abs(row['logFC_Normal_to_AdvCa'])
    # Soft scaling: saturates around logFC = 2
    magnitude_weight = min(overall_fc / 2.0, 1.0)
    
    # D(g) = consistency × magnitude_weight × direction sign
    dg = round(consistency * magnitude_weight, 4) * majority_dir
    return dg

means['D_g'] = means.apply(compute_dg, axis=1)
means['D_g_abs'] = means['D_g'].abs()

# Overall direction label for PCBS compatibility
means['Direction_PCBS'] = 'NS'
means.loc[means['D_g'] > 0, 'Direction_PCBS'] = 'Up'
means.loc[means['D_g'] < 0, 'Direction_PCBS'] = 'Down'

print(f'D(g) score computed for {len(means):,} genes')
print(f'\nD(g) summary:')
print(f'  Genes with |D(g)| > 0   : {(means["D_g_abs"] > 0).sum():,}')
print(f'  Genes with |D(g)| ≥ 0.5 : {(means["D_g_abs"] >= 0.5).sum():,}')
print(f'  Genes with |D(g)| ≥ 0.33: {(means["D_g_abs"] >= 0.33).sum():,}')
print(f'\nDirection (for genes with D(g) ≠ 0):')
print(means['Direction_PCBS'].value_counts().to_string())

## 8. Visualisation

In [ ]:
# ── 8a · D(g) distribution ─────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of D(g) — drop any residual NaN
dg_clean = means['D_g'].dropna()
axes[0].hist(dg_clean, bins=80, color='#3498db', edgecolor='black', lw=0.3, alpha=0.7)
axes[0].axvline(0, color='red', ls='--', lw=1)
axes[0].set_xlabel('D(g) score')
axes[0].set_ylabel('Gene count')
axes[0].set_title('Distribution of Directional Support D(g)')

# Trend category pie
trend_counts = means['Trend'].value_counts()
colors_pie = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#95a5a6']
axes[1].pie(trend_counts.values, labels=trend_counts.index,
            autopct='%1.1f%%', colors=colors_pie[:len(trend_counts)],
            textprops={'fontsize': 8})
axes[1].set_title('Directional Trend Categories')

plt.tight_layout()
plt.savefig('figures/nb4_02_dg_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8b · Expression trajectories for top monotonic genes ──────

# Pick top 5 monotonic up + top 5 monotonic down by |D(g)|
mono_up = means[means['Trend'] == 'Monotonic_Up'].nlargest(5, 'D_g_abs')
mono_dn = means[means['Trend'] == 'Monotonic_Down'].nlargest(5, 'D_g_abs')
top_genes = pd.concat([mono_up, mono_dn])

stages = ['mean_Normal', 'mean_Hyperplasia', 'mean_Cancer_Early', 'mean_Cancer_Advanced']
stage_labels = ['Normal', 'Hyperplasia', 'Early\nCancer', 'Advanced\nCancer']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (trend_name, sub_df), title in zip(
    axes,
    [('up', mono_up), ('down', mono_dn)],
    ['Top 5 Monotonically Increasing', 'Top 5 Monotonically Decreasing']
):
    for gene in sub_df.index:
        vals = sub_df.loc[gene, stages].values
        ax.plot(range(4), vals, 'o-', label=gene, lw=1.5, markersize=5)
    ax.set_xticks(range(4))
    ax.set_xticklabels(stage_labels, fontsize=9)
    ax.set_ylabel('Mean log₂ expression')
    ax.set_title(title)
    ax.legend(fontsize=7, loc='best')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/nb4_03_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8c · Heatmap of top D(g) genes ────────────────────────────

top50 = means.nlargest(50, 'D_g_abs')

# Z-score across stages for heatmap
heatmap_data = top50[stages].copy()
heatmap_data = heatmap_data.subtract(heatmap_data.mean(axis=1), axis=0)
heatmap_data = heatmap_data.divide(heatmap_data.std(axis=1).clip(lower=0.01), axis=0)
heatmap_data.columns = ['Normal', 'Hyperplasia', 'Early Ca', 'Advanced Ca']

fig, ax = plt.subplots(figsize=(6, 14))
sns.heatmap(heatmap_data, cmap='RdBu_r', center=0, ax=ax,
            xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Z-score'})
ax.set_title('Top 50 Genes by |D(g)| — Expression Across Stages')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('figures/nb4_04_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Cross-Reference with NB1/NB2/NB3

Upload the DEG result files from the previous notebooks to check
directional concordance:
- `all_genes_limma_gse48301.csv` (NB1 — PMOS)
- `all_genes_limma_tcga_ucec.csv` (NB2 — Cancer)
- `all_genes_limma_gse106191.csv` (NB3 — Transition)

Upload these files using the Colab file browser (left panel → 📁 → Upload).

In [ ]:
# ── 9 · Load NB1/NB2/NB3 results & check concordance ─────────

def load_if_exists(path, label):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'  ✓ {label}: {len(df):,} genes loaded from {path}')
        return df
    else:
        print(f'  ✗ {label}: not found at {path} — upload it to cross-reference')
        return None

print('Loading previous notebook results...')
nb1 = load_if_exists('all_genes_limma_gse48301.csv',  'NB1 PMOS (GSE48301)')
nb2 = load_if_exists('all_genes_limma_tcga_ucec.csv',  'NB2 Cancer (TCGA-UCEC)')
nb3 = load_if_exists('all_genes_limma_gse106191.csv',  'NB3 Transition (GSE106191)')

In [ ]:
# ── 9b · Concordance analysis ─────────────────────────────────

dir_df = means[['D_g', 'Direction_PCBS', 'Trend',
                'logFC_Normal_to_Hyper', 'logFC_Hyper_to_EarlyCa',
                'logFC_EarlyCa_to_AdvCa', 'logFC_Normal_to_AdvCa']].copy()
dir_df.index.name = 'Gene'
dir_df = dir_df.reset_index()

# Merge with NB1/NB2/NB3 directions
concordance = dir_df.copy()

if nb1 is not None:
    nb1_dir = nb1[['Gene', 'Direction', 'logFC']].rename(
        columns={'Direction': 'Dir_NB1_PMOS', 'logFC': 'logFC_NB1'})
    concordance = concordance.merge(nb1_dir, on='Gene', how='left')

if nb2 is not None:
    nb2_dir = nb2[['Gene', 'Direction', 'logFC']].rename(
        columns={'Direction': 'Dir_NB2_Cancer', 'logFC': 'logFC_NB2'})
    concordance = concordance.merge(nb2_dir, on='Gene', how='left')

if nb3 is not None:
    nb3_dir = nb3[['Gene', 'Direction', 'logFC']].rename(
        columns={'Direction': 'Dir_NB3_Transition', 'logFC': 'logFC_NB3'})
    concordance = concordance.merge(nb3_dir, on='Gene', how='left')

# Count how many notebooks agree with D(g) direction
def count_concordant(row):
    dg_dir = row['Direction_PCBS']
    if dg_dir == 'NS':
        return 0
    n = 0
    total = 0
    for col in ['Dir_NB1_PMOS', 'Dir_NB2_Cancer', 'Dir_NB3_Transition']:
        if col in row.index and pd.notna(row[col]) and row[col] != 'NS':
            total += 1
            if row[col] == dg_dir:
                n += 1
    return n

concordance['n_concordant'] = concordance.apply(count_concordant, axis=1)

# Summary
has_any_nb = any(x is not None for x in [nb1, nb2, nb3])
if has_any_nb:
    print('Concordance with NB1/NB2/NB3 directions:')
    print(concordance['n_concordant'].value_counts().sort_index().to_string())
    
    # Highlight genes concordant across all available notebooks
    n_nbs = sum(1 for x in [nb1, nb2, nb3] if x is not None)
    fully_concordant = concordance[concordance['n_concordant'] == n_nbs]
    print(f'\nFully concordant with all {n_nbs} notebooks: '
          f'{len(fully_concordant)} genes')
    if len(fully_concordant) > 0:
        top_conc = fully_concordant.nlargest(20, 'D_g').head(20) if len(fully_concordant) > 0 else fully_concordant
        print(top_conc[['Gene', 'D_g', 'Direction_PCBS', 'Trend']].to_string(index=False))
else:
    print('No NB1/NB2/NB3 results uploaded — skipping concordance.')
    print('Upload the CSV files and re-run this cell for cross-reference.')

## 10. Known PMOS–EC Bridge Gene Check

In [ ]:
# ── 10 · Known-gene look-up ────────────────────────────────────

KNOWN_GENES = {
    'ESR1':    'Estrogen receptor α',
    'PGR':     'Progesterone receptor',
    'INSR':    'Insulin receptor',
    'PIK3CA':  'PI3K catalytic subunit',
    'AKT1':    'AKT kinase',
    'PTEN':    'Tumour suppressor',
    'TP53':    'p53',
    'CTNNB1':  'β-catenin',
    'KRAS':    'RAS oncogene',
    'SERPINE1': 'PAI-1 (thrombosis / invasion)',
    'CYP19A1': 'Aromatase',
    'MMP9':    'Matrix metalloproteinase',
    'IL6':     'Interleukin-6',
    'TNF':     'TNFα',
    'VEGFA':   'Angiogenesis',
    'MYD88':   'Innate immune signalling',
    'JUN':     'AP-1 transcription factor',
    'CXCL8':   'IL-8 chemokine',
    'IFNG':    'IFN-γ',
    'IL10':    'Anti-inflammatory cytokine',
}

rows = []
for gene, desc in KNOWN_GENES.items():
    if gene in means.index:
        r = means.loc[gene]
        rows.append({
            'Gene': gene, 'Role': desc,
            'logFC_N→H': f"{r['logFC_Normal_to_Hyper']:+.3f}",
            'logFC_H→ECa': f"{r['logFC_Hyper_to_EarlyCa']:+.3f}",
            'logFC_ECa→ACa': f"{r['logFC_EarlyCa_to_AdvCa']:+.3f}",
            'Trend': r['Trend'],
            'D_g': f"{r['D_g']:+.4f}",
        })
    else:
        rows.append({'Gene': gene, 'Role': desc,
                     'logFC_N→H': '—', 'logFC_H→ECa': '—',
                     'logFC_ECa→ACa': '—', 'Trend': 'Not found',
                     'D_g': '—'})

known_df = pd.DataFrame(rows)
known_df.to_csv('results/known_genes_directional_check.csv', index=False)

n_found = (known_df['Trend'] != 'Not found').sum()
n_mono  = known_df['Trend'].str.startswith('Monotonic').sum()
print(f'Known PMOS–EC bridge genes in GSE39099:')
print(f'  Found: {n_found}/{len(KNOWN_GENES)}   Monotonic trend: {n_mono}/{len(KNOWN_GENES)}')
print()
print(known_df.to_string(index=False))

## 11. Save Results

In [ ]:
# ── 11 · Save & summarise ─────────────────────────────────────

# Primary output: directional support for every gene
output = means.reset_index()[[
    'Gene', 'mean_Normal', 'mean_Hyperplasia',
    'mean_Cancer_Early', 'mean_Cancer_Advanced',
    'logFC_Normal_to_Hyper', 'logFC_Hyper_to_EarlyCa',
    'logFC_EarlyCa_to_AdvCa', 'logFC_Normal_to_AdvCa',
    'Trend', 'D_g', 'Direction_PCBS'
]]
output.to_csv('results/directional_support_gse39099.csv', index=False)

# Save concordance table if available
if has_any_nb:
    concordance.to_csv('results/directional_concordance_gse39099.csv', index=False)

n_mono_up = (means['Trend'] == 'Monotonic_Up').sum()
n_mono_dn = (means['Trend'] == 'Monotonic_Down').sum()

print('═' * 60)
print('NOTEBOOK 4 COMPLETE — Directional Support (GSE39099)')
print('═' * 60)
print(f'\nDataset     : GSE39099 (4 pooled arrays, 1 per group, GPL570)')
print(f'              Each array = pool of 10 patients (40 total)')
print(f'Note        : POOLED DATA — fold-changes only, no p-values')
print(f'Genes       : {len(means):,}')
print(f'Monotonic ↑ : {n_mono_up:,}')
print(f'Monotonic ↓ : {n_mono_dn:,}')
print(f'Total mono  : {n_mono_up + n_mono_dn:,}')
print(f'\nKey output  : results/directional_support_gse39099.csv')
print(f'\n→ Next: Notebook 5 (Mendelian Randomisation — causal evidence K(g))')

In [ ]:
# ── (Optional) Download results as ZIP ─────────────────────────
!zip -r nb4_directional_results.zip results/ figures/
try:
    from google.colab import files
    files.download('nb4_directional_results.zip')
except Exception:
    print('Download the ZIP from the file browser on the left.')